# 레슨 11 - 모델 컨텍스트 프로토콜 (MCP)

The **모델 컨텍스트 프로토콜 (MCP)**는 런타임에 에이전트가 도구, 리소스, 데이터 소스를 동적으로 검색하고 사용할 수 있게 해주는 오픈 표준입니다. 에이전트에 도구를 하드코딩하는 대신, MCP는 에이전트가 필요에 따라 기능을 노출하는 외부 서버에 연결할 수 있게 합니다.

In this lesson, you'll learn:
- MCP가 무엇인지와 에이전트 시스템에서 왜 중요한지
- MCP 클라이언트-서버 아키텍처가 어떻게 작동하는지
- MCP 스타일의 도구 검색을 사용하는 에이전트를 구축하는 방법


## 설정

**사전 요구 사항:**
- 배포된 모델이 있는 Azure AI Foundry 프로젝트
- 인증을 위해 `az login`을 실행하세요


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP)이란 무엇인가?

MCP는 AI 에이전트가 외부 도구 및 데이터 소스를 발견하고 상호작용하는 표준 방식을 정의합니다:

- **MCP Server**: 표준 프로토콜을 통해 도구, 리소스 및 프롬프트를 노출합니다
- **MCP Client**: 서버에 연결하고 사용 가능한 기능을 발견하는 에이전트 런타임
- **Dynamic Discovery**: 에이전트는 하드코딩된 도구가 필요하지 않습니다 — 런타임에 사용 가능한 것을 발견합니다

이는 에이전트 코드를 수정하지 않고도 새로운 기능을 추가할 수 있는 확장 가능한 에이전트 시스템을 구축하는 데 매우 유용합니다.


## MCP 작동 방식

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. 에이전트(MCP 클라이언트)가 MCP 서버에 연결합니다
2. 서버는 사용 가능한 도구 목록과 해당 스키마를 응답합니다
3. 그런 다음 에이전트는 추론 과정에서 발견된 어떤 도구든 호출할 수 있습니다
4. 결과는 동일한 프로토콜을 통해 다시 전송됩니다


## MCP 도구 검색 시뮬레이션

실제 MCP 서버는 실행 중인 서버 프로세스를 필요로 하기 때문에, MCP에 연결된 숙박 서비스가 제공할 내용을 시뮬레이션하는 `@tool` 함수들을 사용해 이 패턴을 시연하겠습니다.

실제 운영 환경에서는 이러한 도구들이 로컬에 정의되는 대신 MCP 서버로부터 동적으로 검색될 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP 스타일 도구로 에이전트 구축


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## 프로덕션 환경에서의 MCP

프로덕션 환경에서 MCP는 강력한 패턴을 가능하게 합니다:

- **동적 도구 검색**: 에이전트는 MCP 서버에 연결하여 런타임에 도구를 발견합니다
- **분리된 아키텍처**: 도구 제공자는 에이전트와 독립적으로 업데이트할 수 있습니다
- **조직 간 공유**: 팀은 MCP 서버를 통해 모든 에이전트가 사용할 수 있는 기능을 노출할 수 있습니다
- **Microsoft Agent Framework 지원**: MAF는 `mcp` 통합을 통해 내장된 MCP 클라이언트 지원을 제공합니다

MAF에서 실제 MCP 서버를 사용하려면 `hosted_mcp_tool()` 또는 MCP 클라이언트 통합을 통해 연결합니다.

**자세히 알아보기:**
- [MCP 사양](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework의 MCP 지원](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## 요약

이번 수업에서 배운 내용:
- **MCP**는 에이전트와 도구 제공자 간의 동적 도구 검색을 위한 오픈 표준입니다
- **클라이언트-서버 아키텍처**는 에이전트가 런타임에 기능을 발견할 수 있게 해줍니다
- **MCP**는 도구를 코드 변경 없이 추가할 수 있는 **확장 가능하고 분리된 에이전트 시스템**을 가능하게 합니다
- Microsoft Agent Framework는 프로덕션 환경에서 사용하기 위한 **내장된 MCP 지원**을 제공합니다


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
면책 고지:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 당사는 정확성을 위해 최선을 다하고 있으나 자동 번역에는 오류나 부정확성이 있을 수 있음을 알려드립니다. 원문(원어로 작성된 문서)을 권위 있는 출처로 간주해야 합니다. 중요한 정보의 경우 전문 번역가에 의한 번역을 권장합니다. 이 번역의 사용으로 인해 발생한 오해나 오역에 대해서는 당사가 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
